In [ ]:
"""
Register for vessel cutting.
"""
import pyvista as pv
pv.set_jupyter_backend("html")
pv.start_xvfb()
import numpy as np
import open3d as o3d
import os
import pandas as pd
import argparse
from utils.excision_register import ExcisionRegistrationFromVMTKBranches

suffix = '.obj'
root = "datasets/australian"

label_list = [label for label in os.listdir(root) if os.path.isdir(os.path.join(root, label)) and label != "canonical_typeB"]

for label in label_list:
    print(label)
    cur_root = os.path.join(root, label)
    surface_mesh_file = label + suffix
    surface_mesh_file = os.path.join(cur_root, surface_mesh_file)
    centreline_dir = [c_dir for c_dir in os.listdir(cur_root) if c_dir.startswith('Branches - Centerline model')]
    centreline_dir = os.path.join(cur_root, centreline_dir[0])

    registration = ExcisionRegistrationFromVMTKBranches(surface_mesh_file, centreline_dir, label, cur_root)
    registration.load_checkpoint(chk_path=None, redo=False, branch_retain_length=(4, 4, 6), root_trim_length=0.5)

    control_points_radius_sorted = getattr(registration, "control_points_radius_interpolated_processed")
    radius_inferior = np.mean(control_points_radius_sorted[-2][round(0.5 * len(control_points_radius_sorted[-2])):])
    radius_superior = np.mean(control_points_radius_sorted[-1][round(0.5 * len(control_points_radius_sorted[-1])):])
    radius_root = np.mean(control_points_radius_sorted[0][round(0.5 * len(control_points_radius_sorted[0])):])
    radius_root_in = control_points_radius_sorted[0][-1]

    ratio = np.power(radius_inferior/radius_superior, 2.45)
    ansys_ratio_inferior = ratio / (1 + ratio)
    ansys_ratio_superior = 1 - ansys_ratio_inferior
    print('{} : radius_root: {}; radius_inferior: {}; radius_superior: {}; '.format(label, radius_root, radius_inferior, radius_superior))
    print('{} : ansys_ratio_inferior: {}; ansys_ratio_superior: {}'.format(label, ansys_ratio_inferior, ansys_ratio_superior))
    # registration.load_checkpoint(chk_path=None, redo=False, branch_retain_length=(16*radius_root, 12*radius_inferior, 24*radius_superior), root_trim_length=2.5)




In [ ]:
import vtk
import pyvista as pv
import os

surface_mesh_file = os.path.join(os.getcwd(), 'datasets/australian/005/005.obj')
surface_mesh = pv.read(surface_mesh_file)
p = pv.Plotter()
p.add_mesh(surface_mesh, color='black', opacity=0.025)
p.show()

In [ ]:
import pyvista as pv
# pv.set_jupyter_backend('html')
# pv.start_xvfb()
from pyvista import examples

dataset = examples.download_lucy()
dataset.plot(smooth_shading=True, color='white')

git config --global user.name "WenHaoDing"
git config --global user.email "w.ding23@imperial.ac.uk"


conda install conda-forge::open3d
conda install conda-forge::pandas
conda install -c conda-forge pyvista
conda install conda-forge::scipy
conda install conda-forge::trimesh
conda install conda-forge::trame

pip install 'pyvista[all]'